# BADAS-Open layers
Lists the parts of BADAS-Open and every Linear layer (the layers LoRA can attach to). Runs on the CPU.
Needs the weights in `C:\safe-distance-data\models\badas-open` (`scripts/get_badas_source.py --weights`).

In [1]:
# Cell 1: load BADAS-Open on the CPU (no GPU needed)
import re
import sys
from collections import defaultdict
from pathlib import Path

import torch

BADAS = Path(r"C:\safe-distance-data\models\badas-open")   # downloaded by scripts/get_badas_source.py --weights
sys.path.insert(0, str(BADAS / "src"))
from models.vjepa import VJEPAModel  # noqa: E402

badas = VJEPAModel(model_name="facebook/vjepa2-vitl-fpc16-256-ssv2",
                   checkpoint_path=str(BADAS / "weights" / "badas_open.pth"),
                   device="cpu", frame_count=16, img_size=224, target_fps=8.0, use_sliding_window=False)
badas.load()
net = badas.model
print(type(net).__name__)

d:\My Projects\safe-distance\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading V-JEPA model: facebook/vjepa2-vitl-fpc16-256-ssv2
Using checkpoint: C:\safe-distance-data\models\badas-open\weights\badas_open.pth


Loading weights: 100%|██████████| 587/587 [00:00<00:00, 809.20it/s]
[transformers] VJEPA2Model LOAD REPORT from: facebook/vjepa2-vitl-fpc16-256-ssv2
Key                                                              | Status     |  | 
-----------------------------------------------------------------+------------+--+-
pooler.cross_attention_layer.cross_attn.v_proj.bias              | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.k_proj.weight   | UNEXPECTED |  | 
pooler.cross_attention_layer.cross_attn.k_proj.weight            | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.out_proj.weight | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.self_attn.k_proj.bias     | UNEXPECTED |  | 
pooler.self_attention_layers.{0, 1, 2}.mlp.fc1.weight            | UNEXPECTED |  | 
pooler.cross_attention_layer.mlp.fc2.bias                        | UNEXPECTED |  | 
pooler.cross_attention_layer.layer_norm1.bias                    | UNEXPECTED |  | 
classifier.

Loaded encoder with custom head
Found predictor: VJEPA2Predictor
Using AutoVideoProcessor for V-JEPA 2
EnhancedVideoClassifier


In [2]:
print(net)

EnhancedVideoClassifier(
  (backbone): VJEPA2Model(
    (encoder): VJEPA2Encoder(
      (embeddings): VJEPA2Embeddings(
        (patch_embeddings): VJEPA2PatchEmbeddings3D(
          (proj): Conv3d(3, 1024, kernel_size=(2, 16, 16), stride=(2, 16, 16))
        )
      )
      (layer): ModuleList(
        (0): VJEPA2Layer(
          (norm1): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (attention): VJEPA2RopeAttention(
            (query): Linear(in_features=1024, out_features=1024, bias=True)
            (key): Linear(in_features=1024, out_features=1024, bias=True)
            (value): Linear(in_features=1024, out_features=1024, bias=True)
            (proj): Linear(in_features=1024, out_features=1024, bias=True)
            (dropout): Dropout(p=0.0, inplace=False)
          )
          (drop_path): VJEPA2DropPath(p=0.0)
          (norm2): LayerNorm((1024,), eps=1e-06, elementwise_affine=True)
          (mlp): VJEPA2MLP(
            (fc1): Linear(in_features=1024, ou

In [ ]:
# Cell 2: the big parts, with their parameter counts
def count(m):
    return sum(p.numel() for p in m.parameters())

for name, child in net.named_children():
    print(f"{name:40s} {type(child).__name__:30s} {count(child) / 1e6:8.2f} M")
    for n2, c2 in child.named_children():
        print(f"    {name}.{n2:34s} {type(c2).__name__:30s} {count(c2) / 1e6:8.2f} M")

In [ ]:
# Cell 3: every Linear layer (what LoRA can attach to), grouped: layer numbers replaced by N
groups = defaultdict(list)
for name, m in net.named_modules():
    if isinstance(m, torch.nn.Linear):
        groups[re.sub(r"\.\d+\.", ".N.", name)].append((name, m))
print(f"{'pattern':70s} {'count':>5s}  in -> out")
for pat, items in groups.items():
    m = items[0][1]
    print(f"{pat:70s} {len(items):5d}  {m.in_features} -> {m.out_features}")

In [ ]:
# Cell 4: the full list of Linear layer names (exact names to put in LoRA's target_modules)
for name, m in net.named_modules():
    if isinstance(m, torch.nn.Linear):
        print(name, m.in_features, "->", m.out_features)